# Complete FFA Analysis Workflow

This notebook runs the complete Formal Feature Attribution (FFA) analysis workflow:

1. **FFA Analysis** - Run analysis for all three models (CatBoost, XGBoost, XGBoost RF)
2. **Visualizations** - Generate static visualizations from FFA results
3. **Causal Analysis** - Perform combined causal analysis using probability-based method
4. **Interactive Dashboards** - Create Plotly interactive dashboards
5. **Summary Report** - Generate comprehensive summary of all results

## Outputs Generated:
- FFA analysis results (AXP explanations, feature importance) for each model
- Static visualizations (PNG charts)
- Causal analysis results (CSV + PNG charts)
- Interactive Plotly dashboards (HTML)
- Radar charts for intervention effects
- Summary reports


## Step 1: Setup and Configuration


In [ ]:
import sys
import json
import logging
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "7_ffa_analysis"))

# Configuration
COHORT_NAME = "opioid_ed"
AGE_BAND = "0-12"
AGE_BAND_FNAME = AGE_BAND.replace("-", "_")

# Paths
OUTPUT_DIR = PROJECT_ROOT / '7_ffa_analysis' / 'outputs' / COHORT_NAME / AGE_BAND_FNAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Cohort: {COHORT_NAME}, Age Band: {AGE_BAND}")


## Step 2: Run FFA Analysis for All Three Models


In [ ]:
# Import the FFA analysis script
import subprocess

ffa_script = PROJECT_ROOT / '7_ffa_analysis' / 'run_full_ffa_analysis.py'

print("=" * 80)
print("Running FFA Analysis for All Three Models")
print("=" * 80)

# Run the FFA analysis script
result = subprocess.run(
    [sys.executable, str(ffa_script)],
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT / '7_ffa_analysis')
)

print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

if result.returncode == 0:
    print("\n✓ FFA Analysis completed successfully!")
else:
    print(f"\n✗ FFA Analysis failed with return code {result.returncode}")


## Step 3: Generate Static Visualizations


In [ ]:
viz_script = PROJECT_ROOT / '7_ffa_analysis' / 'create_visualizations.py'

print("=" * 80)
print("Generating Static Visualizations")
print("=" * 80)

result = subprocess.run(
    [sys.executable, str(viz_script)],
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT / '7_ffa_analysis')
)

print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

if result.returncode == 0:
    print("\n✓ Visualizations generated successfully!")
else:
    print(f"\n✗ Visualization generation failed with return code {result.returncode}")


## Step 4: Run Combined Causal Analysis


In [ ]:
causal_script = PROJECT_ROOT / '7_ffa_analysis' / 'combined_causal_analysis.py'

print("=" * 80)
print("Running Combined Causal Analysis")
print("=" * 80)

result = subprocess.run(
    [sys.executable, str(causal_script)],
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT / '7_ffa_analysis')
)

print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

if result.returncode == 0:
    print("\n✓ Causal Analysis completed successfully!")
else:
    print(f"\n✗ Causal Analysis failed with return code {result.returncode}")


## Step 5: Create Interactive Dashboards


In [ ]:
interactive_script = PROJECT_ROOT / '7_ffa_analysis' / 'interactive_risk_explorer.py'

print("=" * 80)
print("Creating Interactive Dashboards")
print("=" * 80)

result = subprocess.run(
    [sys.executable, str(interactive_script)],
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT / '7_ffa_analysis')
)

print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

if result.returncode == 0:
    print("\n✓ Interactive dashboards created successfully!")
else:
    print(f"\n✗ Interactive dashboard creation failed with return code {result.returncode}")


## Step 6: Generate Summary Report


In [ ]:
def generate_summary_report():
    """Generate a comprehensive summary report of all results."""
    
    report_lines = []
    report_lines.append("=" * 80)
    report_lines.append("FFA Analysis Complete Workflow Summary Report")
    report_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report_lines.append(f"Cohort: {COHORT_NAME}, Age Band: {AGE_BAND}")
    report_lines.append("=" * 80)
    report_lines.append("")
    
    # Check FFA analysis results
    report_lines.append("## FFA Analysis Results")
    report_lines.append("-" * 80)
    
    for model_type in ['catboost', 'xgboost', 'xgboost_rf']:
        model_dir = OUTPUT_DIR / model_type
        summary_file = model_dir / 'analysis_summary.json'
        
        if summary_file.exists():
            with open(summary_file, 'r') as f:
                summary = json.load(f)
            
            report_lines.append(f"\n### {model_type.upper()} Model")
            report_lines.append(f"  Coverage Rate: {summary.get('coverage_rate', 'N/A'):.4f}")
            report_lines.append(f"  Total Explanations: {summary.get('total_explanations', 'N/A')}")
            report_lines.append(f"  Unique Features: {summary.get('unique_features', 'N/A')}")
            report_lines.append(f"  Average Rule Length: {summary.get('avg_rule_length', 'N/A'):.2f}")
        else:
            report_lines.append(f"\n### {model_type.upper()} Model: No results found")
    
    # Check visualizations
    report_lines.append("\n## Generated Visualizations")
    report_lines.append("-" * 80)
    
    viz_dir = OUTPUT_DIR / 'visualizations'
    if viz_dir.exists():
        png_files = list(viz_dir.glob('*.png'))
        report_lines.append(f"\nStatic Visualizations (PNG): {len(png_files)} files")
        for png_file in sorted(png_files):
            report_lines.append(f"  - {png_file.name}")
    
    # Check causal analysis results
    report_lines.append("\n## Causal Analysis Results")
    report_lines.append("-" * 80)
    
    causal_dir = OUTPUT_DIR / 'causal_analysis'
    if causal_dir.exists():
        csv_files = list(causal_dir.glob('*.csv'))
        png_files = list(causal_dir.glob('*.png'))
        html_files = list(causal_dir.glob('*.html'))
        
        report_lines.append(f"\nCSV Results: {len(csv_files)} files")
        for csv_file in sorted(csv_files):
            report_lines.append(f"  - {csv_file.name}")
        
        report_lines.append(f"\nCausal Analysis Charts (PNG): {len(png_files)} files")
        for png_file in sorted(png_files):
            report_lines.append(f"  - {png_file.name}")
        
        report_lines.append(f"\nInteractive Charts (HTML): {len(html_files)} files")
        for html_file in sorted(html_files):
            report_lines.append(f"  - {html_file.name}")
    
    # Check interactive dashboards
    report_lines.append("\n## Interactive Dashboards")
    report_lines.append("-" * 80)
    
    interactive_dir = OUTPUT_DIR / 'interactive'
    if interactive_dir.exists():
        html_files = list(interactive_dir.glob('*.html'))
        report_lines.append(f"\nInteractive Dashboards (HTML): {len(html_files)} files")
        for html_file in sorted(html_files):
            report_lines.append(f"  - {html_file.name}")
    
    # Summary statistics
    report_lines.append("\n## Summary Statistics")
    report_lines.append("-" * 80)
    
    # Load top features from causal analysis
    causal_csv = causal_dir / 'causal_importance_probability_method.csv'
    if causal_csv.exists():
        causal_df = pd.read_csv(causal_csv)
        report_lines.append(f"\nTop 10 Causal Features:")
        for idx, row in causal_df.head(10).iterrows():
            report_lines.append(f"  {idx+1}. {row['feature']}: {row['causal_importance']:.6f}")
    
    report_lines.append("\n" + "=" * 80)
    report_lines.append("End of Report")
    report_lines.append("=" * 80)
    
    # Save report
    report_text = "\n".join(report_lines)
    report_file = OUTPUT_DIR / 'complete_workflow_summary.txt'
    
    with open(report_file, 'w') as f:
        f.write(report_text)
    
    print(report_text)
    print(f"\n\nReport saved to: {report_file}")
    
    return report_text

# Generate the summary report
summary = generate_summary_report()


In [ ]:
def display_output_summary():
    """Display a summary of all generated output files."""
    
    print("\n" + "=" * 80)
    print("OUTPUT FILES SUMMARY")
    print("=" * 80)
    
    # FFA Analysis Results
    print("\n1. FFA Analysis Results (per model):")
    for model_type in ['catboost', 'xgboost', 'xgboost_rf']:
        model_dir = OUTPUT_DIR / model_type
        if model_dir.exists():
            files = list(model_dir.glob('*'))
            print(f"   {model_type.upper()}: {len(files)} files")
            for f in sorted(files):
                if f.is_file():
                    print(f"     - {f.name}")
    
    # Visualizations
    print("\n2. Static Visualizations:")
    viz_dir = OUTPUT_DIR / 'visualizations'
    if viz_dir.exists():
        files = list(viz_dir.glob('*.png'))
        print(f"   {len(files)} PNG files")
        for f in sorted(files)[:10]:  # Show first 10
            print(f"     - {f.name}")
        if len(files) > 10:
            print(f"     ... and {len(files) - 10} more")
    
    # Causal Analysis
    print("\n3. Causal Analysis Results:")
    causal_dir = OUTPUT_DIR / 'causal_analysis'
    if causal_dir.exists():
        csv_files = list(causal_dir.glob('*.csv'))
        png_files = list(causal_dir.glob('*.png'))
        html_files = list(causal_dir.glob('*.html'))
        print(f"   CSV files: {len(csv_files)}")
        print(f"   PNG charts: {len(png_files)}")
        print(f"   HTML charts: {len(html_files)}")
    
    # Interactive Dashboards
    print("\n4. Interactive Dashboards:")
    interactive_dir = OUTPUT_DIR / 'interactive'
    if interactive_dir.exists():
        html_files = list(interactive_dir.glob('*.html'))
        print(f"   {len(html_files)} HTML files")
        for f in sorted(html_files):
            print(f"     - {f.name}")
    
    print("\n" + "=" * 80)
    print(f"All outputs saved to: {OUTPUT_DIR}")
    print("=" * 80)

display_output_summary()


In [ ]:
# Load and display top causal features
causal_csv = OUTPUT_DIR / 'causal_analysis' / 'causal_importance_probability_method.csv'

if causal_csv.exists():
    causal_df = pd.read_csv(causal_csv)
    print("\n" + "=" * 80)
    print("TOP 10 CAUSAL FEATURES")
    print("=" * 80)
    print(causal_df.head(10).to_string(index=False))
    print("\n" + "=" * 80)
else:
    print("Causal analysis results not found.")

# Display model coverage rates
print("\n" + "=" * 80)
print("MODEL COVERAGE RATES")
print("=" * 80)

for model_type in ['catboost', 'xgboost', 'xgboost_rf']:
    summary_file = OUTPUT_DIR / model_type / 'analysis_summary.json'
    if summary_file.exists():
        with open(summary_file, 'r') as f:
            summary = json.load(f)
        print(f"{model_type.upper()}: {summary.get('coverage_rate', 0):.4f}")
    else:
        print(f"{model_type.upper()}: No results")

print("=" * 80)
